# Lab 7: Accessing AgentCore Metrics - Runtime, Memory, and Gateway

## Overview

In Labs 1-4, you built a complete Customer Support Agent with:
- **Lab 1**: Core agent functionality with tools
- **Lab 2**: AgentCore Memory for persistent context
- **Lab 3**: AgentCore Gateway for centralized tool management
- **Lab 4**: AgentCore Runtime for production deployment

Now, let's access and monitor the metrics generated by these components. AgentCore automatically emits metrics to CloudWatch for all three services.

## What You'll Learn

- How to query metrics for Runtime, Memory, and Gateway
- Understanding what each metric means
- Creating CloudWatch dashboards for monitoring
- Interpreting metric data for troubleshooting

## Prerequisites

- ✅ Completed Labs 1-4
- ✅ AWS CloudWatch access
- ✅ AgentCore resources deployed and running

## Step 1: Import Libraries and Setup

In [24]:
import boto3
import json
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Import utilities for getting lab resources
from scripts.utils import get_ssm_parameter

# Initialize AWS clients
session = boto3.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
cloudwatch = boto3.client('cloudwatch', region_name=region)

print(f"✅ Connected to AWS Account: {account_id}")
print(f"📍 Region: {region}")
print(f"📊 CloudWatch Namespace: AWS/Bedrock-AgentCore")

✅ Connected to AWS Account: 533267284022
📍 Region: us-east-1
📊 CloudWatch Namespace: AWS/Bedrock-AgentCore


## Step 1.5: Fix Missing Requirements - Diagnostic & Auto-Fix

Let's check and fix all requirements for metrics to appear.

## Step 2: Retrieve Resources from Previous Labs

In [25]:
# Get resources created in previous labs
def get_lab_resources():
    """Retrieve ARNs and IDs from previous labs"""
    resources = {}
    
    # Runtime from Lab 4
    try:
        runtime_arn = get_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
        resources['runtime_arn'] = runtime_arn
        resources['runtime_name'] = runtime_arn.split('/')[-1]
        print(f"✅ Runtime (Lab 4): {runtime_arn}")
    except Exception as e:
        print(f"❌ Runtime not found: Complete Lab 4 first")
        resources['runtime_arn'] = None
    
    # Memory from Lab 2
    try:
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        resources['memory_id'] = memory_id
        resources['memory_arn'] = f"arn:aws:bedrock-agentcore:{region}:{account_id}:memory/{memory_id}"
        print(f"✅ Memory (Lab 2): {memory_id}")
    except Exception as e:
        print(f"❌ Memory not found: Complete Lab 2 first")
        resources['memory_id'] = None
    
    # Gateway from Lab 3
    try:
        gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
        resources['gateway_id'] = gateway_id
        resources['gateway_arn'] = f"arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}"
        print(f"✅ Gateway (Lab 3): {gateway_id}")
    except Exception as e:
        print(f"❌ Gateway not found: Complete Lab 3 first")
        resources['gateway_id'] = None
    
    return resources

# Get all resources
resources = get_lab_resources()
print(f"\n📦 Resources loaded from Labs 1-4")

❌ Runtime not found: Complete Lab 4 first
✅ Memory (Lab 2): CustomerSupportMemory-WcEhTTFp1O
✅ Gateway (Lab 3): customersupport-gw-dcbgswzb5p

📦 Resources loaded from Labs 1-4


## Step 3: Runtime Metrics (Lab 4)

Runtime metrics help you monitor agent performance, errors, and usage patterns.

In [26]:
def query_runtime_metrics(runtime_arn: str, hours_back: int = 24) -> Dict:
    """Query Runtime metrics from CloudWatch"""
    
    if not runtime_arn:
        print("⚠️ No Runtime ARN available. Complete Lab 4 first.")
        return {}
    
    print(f"\n📊 Querying Runtime Metrics")
    print(f"🔗 Resource: {runtime_arn}")
    print("-" * 50)
    
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours_back)
    
    # Runtime metrics according to AgentCore documentation
    metrics_config = [
        ('Invocations', 'Sum', 'Total number of agent invocations'),
        ('Latency', 'Average', 'Average response time in milliseconds'),
        ('SystemErrors', 'Sum', 'Count of system-level errors'),
        ('UserErrors', 'Sum', 'Count of user input errors'),
        ('Throttles', 'Sum', 'Number of throttled requests')
    ]
    
    metrics_data = {}
    
    for metric_name, statistic, description in metrics_config:
        try:
            response = cloudwatch.get_metric_statistics(
                Namespace='AWS/Bedrock-AgentCore',
                MetricName=metric_name,
                Dimensions=[
                    {'Name': 'Resource', 'Value': runtime_arn}
                ],
                StartTime=start_time,
                EndTime=end_time,
                Period=3600,  # 1 hour periods
                Statistics=[statistic]
            )
            
            datapoints = response.get('Datapoints', [])
            if datapoints:
                sorted_points = sorted(datapoints, key=lambda x: x['Timestamp'])
                metrics_data[metric_name] = sorted_points
                latest = sorted_points[-1][statistic]
                print(f"✅ {metric_name}: {latest:.2f}")
                print(f"   {description}")
                print(f"   Data points: {len(datapoints)}")
            else:
                print(f"ℹ️ {metric_name}: No data available")
                print(f"   {description}")
                
        except Exception as e:
            print(f"❌ {metric_name}: Error querying metric - {str(e)}")
    
    return metrics_data

# Query runtime metrics
runtime_metrics = query_runtime_metrics(resources.get('runtime_arn'))

⚠️ No Runtime ARN available. Complete Lab 4 first.


In [27]:
def query_agentcore_by_metric_name(metric_name: str, hours_back: int = 24) -> None:
      """Query a specific metric regardless of dimensions"""

      print(f"\n📊 Querying Metric: {metric_name}")
      print("-" * 50)

      end_time = datetime.now(timezone.utc)
      start_time = end_time - timedelta(hours=hours_back)

      try:
          # First, list all instances of this metric
          response = cloudwatch.list_metrics(
              Namespace='AWS/Bedrock-AgentCore',
              MetricName=metric_name
          )

          if not response['Metrics']:
              print(f"❌ No data found for metric: {metric_name}")
              return

          print(f"✅ Found {len(response['Metrics'])} dimension combinations for {metric_name}\n")

          # Query each dimension combination
          for metric_config in response['Metrics'][:5]:  # Limit to first 5 to avoid too much output
              dimensions = metric_config['Dimensions']
              dim_str = ', '.join([f"{d['Name']}={d['Value']}" for d in dimensions])

              try:
                  stats = cloudwatch.get_metric_statistics(
                      Namespace='AWS/Bedrock-AgentCore',
                      MetricName=metric_name,
                      Dimensions=dimensions,
                      StartTime=start_time,
                      EndTime=end_time,
                      Period=3600,
                      Statistics=['Sum', 'Average', 'Maximum']
                  )

                  if stats['Datapoints']:
                      latest = sorted(stats['Datapoints'], key=lambda x:x['Timestamp'])[-1]
                      print(f"📍 {dim_str}")
                      if 'Sum' in latest:
                          print(f"   Sum: {latest['Sum']:.2f}")
                      if 'Average' in latest:
                          print(f"   Avg: {latest['Average']:.2f}")
                      if 'Maximum' in latest:
                          print(f"   Max: {latest['Maximum']:.2f}")
                      print()

              except Exception as e:
                  print(f"   Error: {str(e)}")

      except Exception as e:
          print(f"❌ Error querying metric: {str(e)}")

# Try common metric names
for metric in ['Invocations', 'Latency', 'Duration', 'Errors', 'SystemErrors']:
    query_agentcore_by_metric_name(metric)


📊 Querying Metric: Invocations
--------------------------------------------------
✅ Found 76 dimension combinations for Invocations


📊 Querying Metric: Latency
--------------------------------------------------
✅ Found 75 dimension combinations for Latency


📊 Querying Metric: Duration
--------------------------------------------------
✅ Found 17 dimension combinations for Duration


📊 Querying Metric: Errors
--------------------------------------------------
✅ Found 5 dimension combinations for Errors

📍 Resource=arn:aws:bedrock-agentcore:us-east-1:533267284022:memory/CustomerSupportMemory-DB1nof41H6, Operation=GetMemory
   Sum: 1.00
   Avg: 1.00
   Max: 1.00

📍 Operation=GetMemory
   Sum: 1.00
   Avg: 1.00
   Max: 1.00


📊 Querying Metric: SystemErrors
--------------------------------------------------
✅ Found 17 dimension combinations for SystemErrors



In [28]:
def query_runtime_metrics_v2(runtime_arn: str, hours_back: int = 24) -> Dict:
      """Query Runtime metrics with proper dimension combinations"""

      if not runtime_arn:
          print("⚠️ No Runtime ARN available.")
          return {}

      print(f"\n📊 Querying Runtime Metrics (Enhanced)")
      print(f"🔗 Resource: {runtime_arn}")
      print("-" * 50)

      end_time = datetime.now(timezone.utc)
      start_time = end_time - timedelta(hours=hours_back)

      metrics_data = {}

      # Try to find metrics with different dimension combinations
      dimension_sets = [
          # Just Resource
          [{'Name': 'Resource', 'Value': runtime_arn}],

          # Resource + Operation for InvokeAgentRuntime
          [
              {'Name': 'Resource', 'Value': runtime_arn},
              {'Name': 'Operation', 'Value': 'InvokeAgentRuntime'}
          ],

          # Resource + Operation + Name (based on your actual data)
          [
              {'Name': 'Resource', 'Value': runtime_arn},
              {'Name': 'Operation', 'Value': 'InvokeAgentRuntime'},
              {'Name': 'Name', 'Value': 'customer_support_agent::DEFAULT'}
          ]
      ]

      metrics_to_query = [
          ('Invocations', 'Sum', 'Total invocations'),
          ('Latency', 'Average', 'Average latency (ms)'),
          ('Errors', 'Sum', 'Total errors'),
          ('SystemErrors', 'Sum', 'System errors'),
          ('Duration', 'Average', 'Average duration')
      ]

      for dimensions in dimension_sets:
          dim_str = ', '.join([f"{d['Name']}={d['Value'].split('/')[-1] if '/' in 
  d['Value'] else d['Value']}" for d in dimensions])
          print(f"\n🔍 Trying dimensions: {dim_str}")

          found_any = False
          for metric_name, statistic, description in metrics_to_query:
              try:
                  response = cloudwatch.get_metric_statistics(
                      Namespace='AWS/Bedrock-AgentCore',
                      MetricName=metric_name,
                      Dimensions=dimensions,
                      StartTime=start_time,
                      EndTime=end_time,
                      Period=3600,
                      Statistics=[statistic]
                  )

                  datapoints = response.get('Datapoints', [])
                  if datapoints:
                      sorted_points = sorted(datapoints, key=lambda x:
  x['Timestamp'])
                      latest = sorted_points[-1][statistic]
                      print(f"   ✅ {metric_name}: {latest:.2f} - {description}")

                      # Store the successful dimension combination
                      if metric_name not in metrics_data:
                          metrics_data[metric_name] = {
                              'dimensions': dimensions,
                              'datapoints': sorted_points,
                              'latest_value': latest
                          }
                      found_any = True

              except Exception as e:
                  pass  # Silently skip failed attempts

          if not found_any:
              print(f"   ℹ️ No data with these dimensions")

      return metrics_data

# Run the enhanced query
runtime_metrics_v2 = query_runtime_metrics_v2(resources.get('runtime_arn'))

⚠️ No Runtime ARN available.


In [29]:
def discover_runtime_operations(runtime_arn: str) -> List[str]:
    """Discover all operations available for a runtime"""

    print(f"\n🔍 Discovering operations for runtime")
    print("-" * 50)

    operations = set()

    try:
        # List all metrics with the Resource dimension matching our runtime
        response = cloudwatch.list_metrics(
            Namespace='AWS/Bedrock-AgentCore',
            Dimensions=[
                {'Name': 'Resource', 'Value': runtime_arn}
            ]
        )

        for metric in response['Metrics']:
            for dim in metric['Dimensions']:
                if dim['Name'] == 'Operation':
                    operations.add(dim['Value'])

        if operations:
            print(f"✅ Found {len(operations)} operations:")
            for op in sorted(operations):
                print(f"   • {op}")
        else:
            print("ℹ️ No operations found for this runtime")

    except Exception as e:
        print(f"❌ Error: {str(e)}")

    return list(operations)

# Discover operations
operations = discover_runtime_operations(resources.get('runtime_arn'))


🔍 Discovering operations for runtime
--------------------------------------------------
❌ Error: Parameter validation failed:
Invalid type for parameter Dimensions[0].Value, value: None, type: <class 'NoneType'>, valid types: <class 'str'>


In [20]:
def query_all_runtime_metrics(runtime_arn: str, hours_back: int = 24) -> Dict:
    """Query ALL available metrics for a runtime"""

    if not runtime_arn:
        print("⚠️ No Runtime ARN available.")
        return {}

    print(f"\n📊 Querying ALL Runtime Metrics")
    print(f"🔗 Resource: {runtime_arn}")
    print("-" * 50)

    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours_back)

    # First, find all metric/dimension combinations for this runtime
    all_metrics = {}

    try:
        response = cloudwatch.list_metrics(
            Namespace='AWS/Bedrock-AgentCore',
            Dimensions=[
                {'Name': 'Resource', 'Value': runtime_arn}
            ]
        )

        print(f"\n✅ Found {len(response['Metrics'])} metric combinations\n")

        # Group by metric name
        for metric_config in response['Metrics']:
            metric_name = metric_config['MetricName']
            dimensions = metric_config['Dimensions']

            if metric_name not in all_metrics:
                all_metrics[metric_name] = []

            all_metrics[metric_name].append(dimensions)

        # Query each unique metric
        results = {}
        for metric_name, dimension_sets in all_metrics.items():
            print(f"📈 {metric_name}:")

            for dimensions in dimension_sets[:3]:  # Limit to first 3 combinations
                dim_str = ', '.join([f"{d['Name']}={d['Value'].split('/')[-1] if '/' in d['Value'] else d['Value']}" for d in dimensions if d['Name'] !='Resource'])

                try:
                    stats = cloudwatch.get_metric_statistics(
                        Namespace='AWS/Bedrock-AgentCore',
                        MetricName=metric_name,
                        Dimensions=dimensions,
                        StartTime=start_time,
                        EndTime=end_time,
                        Period=3600,
                        Statistics=['Sum', 'Average', 'SampleCount']
                    )

                    if stats['Datapoints']:
                        latest = sorted(stats['Datapoints'], key=lambda x:x['Timestamp'])[-1]

                        output = f"   • {dim_str if dim_str else 'Base metric'}: "
                        if 'Sum' in latest:
                            output += f"Sum={latest['Sum']:.0f} "
                        if 'Average' in latest:
                            output += f"Avg={latest['Average']:.1f} "
                        if 'SampleCount' in latest:
                            output += f"Count={latest['SampleCount']:.0f}"

                        print(output)

                        if metric_name not in results:
                            results[metric_name] = []
                        results[metric_name].append({
                            'dimensions': dimensions,
                            'latest': latest
                        })

                except Exception as e:
                    pass  # Skip errors silently

            if len(dimension_sets) > 3:
                print(f"   ... and {len(dimension_sets) - 3} more combinations")
            print()

    except Exception as e:
        print(f"❌ Error: {str(e)}")

    return results

# Query all metrics
all_metrics = query_all_runtime_metrics(resources.get('runtime_arn'))



📊 Querying ALL Runtime Metrics
🔗 Resource: arn:aws:bedrock-agentcore:us-east-1:533267284022:runtime/customer_support_agent-b0Ilb5ACG7
--------------------------------------------------

✅ Found 8 metric combinations

📈 Throttles:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT: Sum=0 Avg=0.0 Count=3

📈 Duration:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT: Sum=47156 Avg=15718.7 Count=3

📈 UserErrors:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT: Sum=0 Avg=0.0 Count=3

📈 SystemErrors:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT: Sum=0 Avg=0.0 Count=3

📈 Invocations:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT: Sum=3 Avg=1.0 Count=3

📈 Latency:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT: Sum=47156 Avg=15718.7 Count=3

📈 Sessions:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT: Sum=2 Avg=1.0 Count=2

📈

## Step 4: Memory Metrics (Lab 2)

Memory metrics show how your agent's memory is being used and accessed.

In [30]:
def query_memory_metrics_v2(memory_arn: str, memory_id: str, hours_back: int = 24) -> Dict:
    """Query Memory metrics with proper dimension combinations"""

    if not memory_arn:
        print("⚠️ No Memory ARN available.")
        return {}

    print(f"\n📊 Querying Memory Metrics (Enhanced)")
    print(f"🔗 Resource: {memory_arn}")
    print(f"🆔 Memory ID: {memory_id}")
    print("-" * 50)

    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours_back)

    metrics_data = {}

    # Memory operations typically include: CreateSession, ListMemory, CreateEvent, GetEvent
    dimension_sets = [
        # Just Resource
        [{'Name': 'Resource', 'Value': memory_arn}],

        # Resource + Operation for memory operations
        [
            {'Name': 'Resource', 'Value': memory_arn},
            {'Name': 'Operation', 'Value': 'CreateSession'}
        ],
        [
            {'Name': 'Resource', 'Value': memory_arn},
            {'Name': 'Operation', 'Value': 'ListMemory'}
        ],
        [
            {'Name': 'Resource', 'Value': memory_arn},
            {'Name': 'Operation', 'Value': 'CreateEvent'}
        ],
        [
            {'Name': 'Resource', 'Value': memory_arn},
            {'Name': 'Operation', 'Value': 'GetEvent'}
        ],

        # Try with ItemType dimension (for memory items)
        [
            {'Name': 'ItemType', 'Value': 'Memory'},
            {'Name': 'Resource', 'Value': memory_arn}
        ]
    ]

    metrics_to_query = [
        ('Invocations', 'Sum', 'Total memory operations'),
        ('Latency', 'Average', 'Average latency (ms)'),
        ('Duration', 'Average', 'Average duration (ms)'),
        ('CreationCount', 'Sum', 'Items created'),
        ('Sessions', 'Sum', 'Sessions created'),
        ('Errors', 'Sum', 'Total errors')
    ]

    for dimensions in dimension_sets:
        dim_str = ', '.join([f"{d['Name']}={d['Value'].split('/')[-1] if '/' in d['Value'] else d['Value']}" for d in dimensions])
        print(f"\n🔍 Trying dimensions: {dim_str}")

        found_any = False
        for metric_name, statistic, description in metrics_to_query:
            try:
                response = cloudwatch.get_metric_statistics(
                    Namespace='AWS/Bedrock-AgentCore',
                    MetricName=metric_name,
                    Dimensions=dimensions,
                    StartTime=start_time,
                    EndTime=end_time,
                    Period=3600,
                    Statistics=[statistic]
                )

                datapoints = response.get('Datapoints', [])
                if datapoints:
                    sorted_points = sorted(datapoints, key=lambda x: x['Timestamp'])
                    latest = sorted_points[-1][statistic]
                    print(f"   ✅ {metric_name}: {latest:.2f} - {description}")

                    if metric_name not in metrics_data:
                        metrics_data[metric_name] = {
                            'dimensions': dimensions,
                            'datapoints': sorted_points,
                            'latest_value': latest
                        }
                    found_any = True

            except Exception as e:
                pass

        if not found_any:
            print(f"   ℹ️ No data with these dimensions")

    return metrics_data

def discover_memory_operations(memory_arn: str) -> List[str]:
    """Discover all operations available for memory"""

    print(f"\n🔍 Discovering operations for memory")
    print("-" * 50)

    operations = set()

    try:
        # List all metrics with the Resource dimension matching our memory
        response = cloudwatch.list_metrics(
            Namespace='AWS/Bedrock-AgentCore',
            Dimensions=[
                {'Name': 'Resource', 'Value': memory_arn}
            ]
        )

        for metric in response['Metrics']:
            for dim in metric['Dimensions']:
                if dim['Name'] == 'Operation':
                    operations.add(dim['Value'])

        if operations:
            print(f"✅ Found {len(operations)} operations:")
            for op in sorted(operations):
                print(f"   • {op}")
        else:
            # Try without Resource filter to see all memory-related operations
            response = cloudwatch.list_metrics(
                Namespace='AWS/Bedrock-AgentCore',
                MetricName='Invocations'
            )

            for metric in response['Metrics']:
                for dim in metric['Dimensions']:
                    if dim['Name'] == 'Operation' and 'Memory' in dim['Value']:
                        operations.add(dim['Value'])

            if operations:
                print(f"✅ Found {len(operations)} memory-related operations (global):")
                for op in sorted(operations):
                    print(f"   • {op}")

    except Exception as e:
        print(f"❌ Error: {str(e)}")

    return list(operations)

# Usage:
if resources.get('memory_id'):
    memory_operations = discover_memory_operations(resources.get('memory_arn'))
    memory_metrics_v2 = query_memory_metrics_v2(
        resources.get('memory_arn'),
        resources.get('memory_id')
    )



🔍 Discovering operations for memory
--------------------------------------------------
✅ Found 6 operations:
   • Consolidation
   • CreateEvent
   • CreateMemory
   • Extraction
   • GetMemory
   • RetrieveMemoryRecords

📊 Querying Memory Metrics (Enhanced)
🔗 Resource: arn:aws:bedrock-agentcore:us-east-1:533267284022:memory/CustomerSupportMemory-WcEhTTFp1O
🆔 Memory ID: CustomerSupportMemory-WcEhTTFp1O
--------------------------------------------------

🔍 Trying dimensions: Resource=CustomerSupportMemory-WcEhTTFp1O
   ℹ️ No data with these dimensions

🔍 Trying dimensions: Resource=CustomerSupportMemory-WcEhTTFp1O, Operation=CreateSession
   ℹ️ No data with these dimensions

🔍 Trying dimensions: Resource=CustomerSupportMemory-WcEhTTFp1O, Operation=ListMemory
   ℹ️ No data with these dimensions

🔍 Trying dimensions: Resource=CustomerSupportMemory-WcEhTTFp1O, Operation=CreateEvent
   ✅ Invocations: 5.00 - Total memory operations
   ✅ Latency: 59.80 - Average latency (ms)

🔍 Trying dimens

In [31]:
def query_memory_metrics(memory_arn: str, hours_back: int = 24) -> Dict:
    """Query Memory metrics from CloudWatch"""
    
    if not memory_arn:
        print("⚠️ No Memory ARN available. Complete Lab 2 first.")
        return {}
    
    print(f"\n📊 Querying Memory Metrics")
    print(f"🔗 Resource: {memory_arn}")
    print("-" * 50)
    
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours_back)
    
    # Memory metrics
    metrics_config = [
        ('CreationCount', 'Sum', 'Number of memory creation events'),
        ('Sessions', 'Sum', 'Number of active memory sessions'),
        ('Duration', 'Average', 'Average duration of memory operations')
    ]
    
    metrics_data = {}
    
    for metric_name, statistic, description in metrics_config:
        try:
            response = cloudwatch.get_metric_statistics(
                Namespace='AWS/Bedrock-AgentCore',
                MetricName=metric_name,
                Dimensions=[
                    {'Name': 'Resource', 'Value': memory_arn}
                ],
                StartTime=start_time,
                EndTime=end_time,
                Period=3600,
                Statistics=[statistic]
            )
            
            datapoints = response.get('Datapoints', [])
            if datapoints:
                sorted_points = sorted(datapoints, key=lambda x: x['Timestamp'])
                metrics_data[metric_name] = sorted_points
                latest = sorted_points[-1][statistic]
                print(f"✅ {metric_name}: {latest:.2f}")
                print(f"   {description}")
                print(f"   Data points: {len(datapoints)}")
            else:
                print(f"ℹ️ {metric_name}: No data available")
                print(f"   {description}")
                
        except Exception as e:
            print(f"❌ {metric_name}: Error querying metric - {str(e)}")
    
    return metrics_data

# Query memory metrics
if resources.get('memory_id'):
    memory_metrics = query_memory_metrics(resources.get('memory_arn'))
else:
    memory_metrics = {}


📊 Querying Memory Metrics
🔗 Resource: arn:aws:bedrock-agentcore:us-east-1:533267284022:memory/CustomerSupportMemory-WcEhTTFp1O
--------------------------------------------------
ℹ️ CreationCount: No data available
   Number of memory creation events
ℹ️ Sessions: No data available
   Number of active memory sessions
ℹ️ Duration: No data available
   Average duration of memory operations


## Step 5: Gateway Metrics (Lab 3)

Gateway metrics track tool usage and performance.

In [32]:
def query_gateway_metrics_v2(gateway_arn: str, gateway_id: str, hours_back: int = 24) -> Dict:
    """Query Gateway metrics with proper dimension combinations"""

    if not gateway_arn:
        print("⚠️ No Gateway ARN available.")
        return {}

    print(f"\n📊 Querying Gateway Metrics (Enhanced)")
    print(f"🔗 Resource: {gateway_arn}")
    print(f"🆔 Gateway ID: {gateway_id}")
    print("-" * 50)

    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours_back)

    metrics_data = {}

    # Gateway operations and target types
    dimension_sets = [
        # Just Resource
        [{'Name': 'Resource', 'Value': gateway_arn}],

        # Resource + Operation for gateway operations
        [
            {'Name': 'Resource', 'Value': gateway_arn},
            {'Name': 'Operation', 'Value': 'InvokeTarget'}
        ],
        [
            {'Name': 'Resource', 'Value': gateway_arn},
            {'Name': 'Operation', 'Value': 'ListTargets'}
        ],

        # Resource + TargetType (for Lambda functions)
        [
            {'Name': 'Resource', 'Value': gateway_arn},
            {'Name': 'TargetType', 'Value': 'LAMBDA'}
        ],

        # Try with specific target names if available
        [
            {'Name': 'Resource', 'Value': gateway_arn},
            {'Name': 'TargetType', 'Value': 'LAMBDA'},
            {'Name': 'TargetName', 'Value': 'OrderLookup'}  # Example target
        ]
    ]

    metrics_to_query = [
        ('Invocations', 'Sum', 'Total gateway invocations'),
        ('Latency', 'Average', 'Average latency (ms)'),
        ('TargetExecutionTime', 'Average', 'Target execution time (ms)'),
        ('TargetType.LAMBDA', 'Sum', 'Lambda invocations'),
        ('Errors', 'Sum', 'Total errors'),
        ('Throttles', 'Sum', 'Throttled requests'),
        ('Duration', 'Average', 'Average duration (ms)')
    ]

    for dimensions in dimension_sets:
        dim_str = ', '.join([f"{d['Name']}={d['Value'].split('/')[-1] if '/' in d['Value'] else d['Value']}" for d in dimensions])
        print(f"\n🔍 Trying dimensions: {dim_str}")

        found_any = False
        for metric_name, statistic, description in metrics_to_query:
            try:
                response = cloudwatch.get_metric_statistics(
                    Namespace='AWS/Bedrock-AgentCore',
                    MetricName=metric_name,
                    Dimensions=dimensions,
                    StartTime=start_time,
                    EndTime=end_time,
                    Period=3600,
                    Statistics=[statistic]
                )

                datapoints = response.get('Datapoints', [])
                if datapoints:
                    sorted_points = sorted(datapoints, key=lambda x: x['Timestamp'])
                    latest = sorted_points[-1][statistic]
                    print(f"   ✅ {metric_name}: {latest:.2f} - {description}")

                    if metric_name not in metrics_data:
                        metrics_data[metric_name] = {
                            'dimensions': dimensions,
                            'datapoints': sorted_points,
                            'latest_value': latest
                        }
                    found_any = True

            except Exception as e:
                pass

        if not found_any:
            print(f"   ℹ️ No data with these dimensions")

    return metrics_data

def discover_gateway_targets(gateway_arn: str) -> Dict:
    """Discover all targets and operations for gateway"""

    print(f"\n🔍 Discovering targets for gateway")
    print("-" * 50)

    targets = {'operations': set(), 'target_types': set(), 'target_names': set()}

    try:
        # List all metrics with the Resource dimension matching our gateway
        response = cloudwatch.list_metrics(
            Namespace='AWS/Bedrock-AgentCore',
            Dimensions=[
                {'Name': 'Resource', 'Value': gateway_arn}
            ]
        )

        for metric in response['Metrics']:
            for dim in metric['Dimensions']:
                if dim['Name'] == 'Operation':
                    targets['operations'].add(dim['Value'])
                elif dim['Name'] == 'TargetType':
                    targets['target_types'].add(dim['Value'])
                elif dim['Name'] == 'TargetName':
                    targets['target_names'].add(dim['Value'])

        if targets['operations']:
            print(f"✅ Operations:")
            for op in sorted(targets['operations']):
                print(f"   • {op}")

        if targets['target_types']:
            print(f"\n✅ Target Types:")
            for tt in sorted(targets['target_types']):
                print(f"   • {tt}")

        if targets['target_names']:
            print(f"\n✅ Target Names:")
            for tn in sorted(targets['target_names']):
                print(f"   • {tn}")

        if not any(targets.values()):
            print("ℹ️ No targets found for this gateway")

    except Exception as e:
        print(f"❌ Error: {str(e)}")

    return {k: list(v) for k, v in targets.items()}

# Usage:
if resources.get('gateway_id'):
    gateway_targets = discover_gateway_targets(resources.get('gateway_arn'))
    gateway_metrics_v2 = query_gateway_metrics_v2(
        resources.get('gateway_arn'),
        resources.get('gateway_id')
    )



🔍 Discovering targets for gateway
--------------------------------------------------
✅ Operations:
   • CallToolMcp
   • InitializeMcp
   • InitializedNotificationMcp
   • ListToolsMcp

📊 Querying Gateway Metrics (Enhanced)
🔗 Resource: arn:aws:bedrock-agentcore:us-east-1:533267284022:gateway/customersupport-gw-dcbgswzb5p
🆔 Gateway ID: customersupport-gw-dcbgswzb5p
--------------------------------------------------

🔍 Trying dimensions: Resource=customersupport-gw-dcbgswzb5p
   ℹ️ No data with these dimensions

🔍 Trying dimensions: Resource=customersupport-gw-dcbgswzb5p, Operation=InvokeTarget
   ℹ️ No data with these dimensions

🔍 Trying dimensions: Resource=customersupport-gw-dcbgswzb5p, Operation=ListTargets
   ℹ️ No data with these dimensions

🔍 Trying dimensions: Resource=customersupport-gw-dcbgswzb5p, TargetType=LAMBDA
   ℹ️ No data with these dimensions

🔍 Trying dimensions: Resource=customersupport-gw-dcbgswzb5p, TargetType=LAMBDA, TargetName=OrderLookup
   ℹ️ No data with thes

In [5]:
def query_gateway_metrics(gateway_arn: str, hours_back: int = 24) -> Dict:
    """Query Gateway metrics from CloudWatch"""
    
    if not gateway_arn:
        print("⚠️ No Gateway ARN available. Complete Lab 3 first.")
        return {}
    
    print(f"\n📊 Querying Gateway Metrics")
    print(f"🔗 Resource: {gateway_arn}")
    print("-" * 50)
    
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours_back)
    
    # Gateway metrics
    metrics_config = [
        ('Invocations', 'Sum', 'Total gateway invocations'),
        ('TargetExecutionTime', 'Average', 'Average time for target execution'),
        ('TargetType.LAMBDA', 'Sum', 'Number of Lambda target invocations'),
        ('Errors', 'Sum', 'Total gateway errors'),
        ('Latency', 'Average', 'Gateway latency in milliseconds')
    ]
    
    metrics_data = {}
    
    for metric_name, statistic, description in metrics_config:
        try:
            response = cloudwatch.get_metric_statistics(
                Namespace='AWS/Bedrock-AgentCore',
                MetricName=metric_name,
                Dimensions=[
                    {'Name': 'Resource', 'Value': gateway_arn}
                ],
                StartTime=start_time,
                EndTime=end_time,
                Period=3600,
                Statistics=[statistic]
            )
            
            datapoints = response.get('Datapoints', [])
            if datapoints:
                sorted_points = sorted(datapoints, key=lambda x: x['Timestamp'])
                metrics_data[metric_name] = sorted_points
                latest = sorted_points[-1][statistic]
                print(f"✅ {metric_name}: {latest:.2f}")
                print(f"   {description}")
                print(f"   Data points: {len(datapoints)}")
            else:
                print(f"ℹ️ {metric_name}: No data available")
                print(f"   {description}")
                
        except Exception as e:
            print(f"❌ {metric_name}: Error querying metric - {str(e)}")
    
    return metrics_data

# Query gateway metrics
if resources.get('gateway_id'):
    gateway_metrics = query_gateway_metrics(resources.get('gateway_arn'))
else:
    gateway_metrics = {}


📊 Querying Gateway Metrics
🔗 Resource: arn:aws:bedrock-agentcore:us-east-1:533267284022:gateway/customersupport-gw-hjwbfb7iep
--------------------------------------------------
ℹ️ Invocations: No data available
   Total gateway invocations
ℹ️ TargetExecutionTime: No data available
   Average time for target execution
ℹ️ TargetType.LAMBDA: No data available
   Number of Lambda target invocations
ℹ️ Errors: No data available
   Total gateway errors
ℹ️ Latency: No data available
   Gateway latency in milliseconds


## Step 6: Create Runtime Dashboard

In [6]:
def create_runtime_dashboard(runtime_arn: str, runtime_name: str) -> bool:
    """Create CloudWatch dashboard for Runtime metrics"""
    
    if not runtime_arn:
        print("⚠️ No Runtime ARN available")
        return False
    
    dashboard_name = f"AgentCore-Runtime-{runtime_name}"
    
    dashboard_body = {
        "widgets": [
            {
                "type": "metric",
                "x": 0, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "title": "Runtime Performance",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "Invocations", "Resource", runtime_arn, {"stat": "Sum"}],
                        [".", "Latency", ".", ".", {"stat": "Average", "yAxis": "right"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {
                        "left": {"label": "Invocations"},
                        "right": {"label": "Latency (ms)"}
                    }
                }
            },
            {
                "type": "metric",
                "x": 12, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "title": "Runtime Errors",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "SystemErrors", "Resource", runtime_arn, {"stat": "Sum", "color": "#d62728"}],
                        [".", "UserErrors", ".", ".", {"stat": "Sum", "color": "#ff7f0e"}],
                        [".", "Throttles", ".", ".", {"stat": "Sum", "color": "#9467bd"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Error Count"}}
                }
            }
        ]
    }
    
    try:
        response = cloudwatch.put_dashboard(
            DashboardName=dashboard_name,
            DashboardBody=json.dumps(dashboard_body)
        )
        
        dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
        print(f"✅ Created Runtime Dashboard: {dashboard_name}")
        print(f"📊 View it here: {dashboard_url}")
        return True
        
    except Exception as e:
        print(f"❌ Error creating dashboard: {str(e)}")
        return False

# Create runtime dashboard
if resources.get('runtime_arn'):
    create_runtime_dashboard(resources['runtime_arn'], resources['runtime_name'])

✅ Created Runtime Dashboard: AgentCore-Runtime-customer_support_agent-b0Ilb5ACG7
📊 View it here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=AgentCore-Runtime-customer_support_agent-b0Ilb5ACG7


## Step 7: Create Memory Dashboard

In [7]:
def create_memory_dashboard(memory_arn: str, memory_id: str) -> bool:
    """Create CloudWatch dashboard for Memory metrics"""
    
    if not memory_arn:
        print("⚠️ No Memory ARN available")
        return False
    
    dashboard_name = f"AgentCore-Memory-{memory_id}"
    
    dashboard_body = {
        "widgets": [
            {
                "type": "metric",
                "x": 0, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "title": "Memory Usage",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "CreationCount", "Resource", memory_arn, {"stat": "Sum"}],
                        [".", "Sessions", ".", ".", {"stat": "Sum", "yAxis": "right"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {
                        "left": {"label": "Creation Count"},
                        "right": {"label": "Sessions"}
                    }
                }
            },
            {
                "type": "metric",
                "x": 12, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "title": "Memory Performance",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "Duration", "Resource", memory_arn, {"stat": "Average"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Duration (ms)"}}
                }
            }
        ]
    }
    
    try:
        response = cloudwatch.put_dashboard(
            DashboardName=dashboard_name,
            DashboardBody=json.dumps(dashboard_body)
        )
        
        dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
        print(f"✅ Created Memory Dashboard: {dashboard_name}")
        print(f"📊 View it here: {dashboard_url}")
        return True
        
    except Exception as e:
        print(f"❌ Error creating dashboard: {str(e)}")
        return False

# Create memory dashboard
if resources.get('memory_id'):
    create_memory_dashboard(resources['memory_arn'], resources['memory_id'])

✅ Created Memory Dashboard: AgentCore-Memory-CustomerSupportMemory-DB1nof41H6
📊 View it here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=AgentCore-Memory-CustomerSupportMemory-DB1nof41H6


## Step 8: Create Gateway Dashboard

In [8]:
def create_gateway_dashboard(gateway_arn: str, gateway_id: str) -> bool:
    """Create CloudWatch dashboard for Gateway metrics"""
    
    if not gateway_arn:
        print("⚠️ No Gateway ARN available")
        return False
    
    dashboard_name = f"AgentCore-Gateway-{gateway_id}"
    
    dashboard_body = {
        "widgets": [
            {
                "type": "metric",
                "x": 0, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "title": "Gateway Performance",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "Invocations", "Resource", gateway_arn, {"stat": "Sum"}],
                        [".", "TargetExecutionTime", ".", ".", {"stat": "Average", "yAxis": "right"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {
                        "left": {"label": "Invocations"},
                        "right": {"label": "Execution Time (ms)"}
                    }
                }
            },
            {
                "type": "metric",
                "x": 12, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "title": "Gateway Target Types",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "TargetType.LAMBDA", "Resource", gateway_arn, {"stat": "Sum"}],
                        [".", "Errors", ".", ".", {"stat": "Sum", "color": "#d62728"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Count"}}
                }
            }
        ]
    }
    
    try:
        response = cloudwatch.put_dashboard(
            DashboardName=dashboard_name,
            DashboardBody=json.dumps(dashboard_body)
        )
        
        dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
        print(f"✅ Created Gateway Dashboard: {dashboard_name}")
        print(f"📊 View it here: {dashboard_url}")
        return True
        
    except Exception as e:
        print(f"❌ Error creating dashboard: {str(e)}")
        return False

# Create gateway dashboard
if resources.get('gateway_id'):
    create_gateway_dashboard(resources['gateway_arn'], resources['gateway_id'])

✅ Created Gateway Dashboard: AgentCore-Gateway-customersupport-gw-hjwbfb7iep
📊 View it here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=AgentCore-Gateway-customersupport-gw-hjwbfb7iep


## Step 9: Summary and Next Steps

In [9]:
print("\n" + "="*60)
print("📊 AgentCore Metrics Summary")
print("="*60)

print("\n✅ Available Components:")
if resources.get('runtime_arn'):
    print(f"   • Runtime: {resources['runtime_name']}")
if resources.get('memory_id'):
    print(f"   • Memory: {resources['memory_id']}")
if resources.get('gateway_id'):
    print(f"   • Gateway: {resources['gateway_id']}")

print("\n📈 Metrics Status:")
print(f"   • Runtime: {'Data available' if runtime_metrics else 'No data yet'}")
print(f"   • Memory: {'Data available' if memory_metrics else 'No data yet'}")
print(f"   • Gateway: {'Data available' if gateway_metrics else 'No data yet'}")

print("\n🔧 To Generate Metrics:")
print("   1. Enable observability for each resource in the AgentCore console")
print("   2. Invoke your agent to generate activity")
print("   3. Wait 2-5 minutes for metrics to appear")
print("   4. Re-run this notebook to see updated data")

print("\n📊 CloudWatch Namespace: AWS/Bedrock-AgentCore")
print("\n🎉 Your metrics monitoring is now configured!")


📊 AgentCore Metrics Summary

✅ Available Components:
   • Runtime: customer_support_agent-b0Ilb5ACG7
   • Memory: CustomerSupportMemory-DB1nof41H6
   • Gateway: customersupport-gw-hjwbfb7iep

📈 Metrics Status:
   • Runtime: No data yet
   • Memory: No data yet
   • Gateway: No data yet

🔧 To Generate Metrics:
   1. Enable observability for each resource in the AgentCore console
   2. Invoke your agent to generate activity
   3. Wait 2-5 minutes for metrics to appear
   4. Re-run this notebook to see updated data

📊 CloudWatch Namespace: AWS/Bedrock-AgentCore

🎉 Your metrics monitoring is now configured!


## Understanding the Metrics

### Runtime Metrics
- **Invocations**: Total number of times your agent was called
- **Latency**: How long each invocation takes to complete
- **SystemErrors**: Errors from the AgentCore system
- **UserErrors**: Errors from invalid user inputs
- **Throttles**: Requests rejected due to rate limiting

### Memory Metrics
- **CreationCount**: New memory entries created
- **Sessions**: Active memory sessions
- **Duration**: Time taken for memory operations

### Gateway Metrics
- **Invocations**: Total gateway tool calls
- **TargetExecutionTime**: Time taken by Lambda functions
- **TargetType.LAMBDA**: Count of Lambda invocations
- **Errors**: Gateway-level errors
- **Latency**: Gateway processing time

## Troubleshooting

If you don't see metrics:
1. Verify observability is enabled for your resources
2. Check that you've invoked your agent recently
3. Ensure you have the correct IAM permissions for CloudWatch
4. Confirm the namespace is `AWS/Bedrock-AgentCore`
5. Wait a few minutes - metrics can take time to appear

## Congratulations! ✅

You've successfully set up metrics monitoring for your AgentCore Runtime, Memory, and Gateway components!


  These functions will help you:
  1. Discover what metrics and operations are actually available
  2. Query them with the correct dimension combinations
  3. Handle cases where metrics might not use the Resource dimension directly

  The key insight is that AgentCore metrics often use operation-specific dimensions
  rather than just the resource ARN.

In [23]:
# Here's a comprehensive function that works for any AgentCore resource:

def discover_all_metrics_for_resource(resource_arn: str, resource_type: str = "Unknown") -> Dict:
    """Discover ALL available metrics and dimensions for any AgentCore resource"""

    print(f"\n🔍 Discovering all metrics for {resource_type}")
    print(f"🔗 Resource: {resource_arn}")
    print("-" * 50)

    discovered = {
        'metrics': {},
        'operations': set(),
        'dimensions': set()
    }

    try:
        # Find all metrics for this resource
        response = cloudwatch.list_metrics(
            Namespace='AWS/Bedrock-AgentCore',
            Dimensions=[
                {'Name': 'Resource', 'Value': resource_arn}
            ]
        )

        if not response['Metrics']:
            # Try without Resource filter but look for related operations
            print("ℹ️ No metrics found with Resource dimension, searching globally...")

            response = cloudwatch.list_metrics(
                Namespace='AWS/Bedrock-AgentCore'
            )

            # Filter for relevant metrics based on resource type
            keyword = resource_type.lower()
            relevant_metrics = []
            for metric in response['Metrics']:
                for dim in metric['Dimensions']:
                    if keyword in dim['Value'].lower():
                        relevant_metrics.append(metric)
                        break

            response['Metrics'] = relevant_metrics

        print(f"✅ Found {len(response['Metrics'])} metric combinations\n")

        # Analyze the metrics
        for metric in response['Metrics']:
            metric_name = metric['MetricName']

            if metric_name not in discovered['metrics']:
                discovered['metrics'][metric_name] = []

            dim_combo = {}
            for dim in metric['Dimensions']:
                dim_combo[dim['Name']] = dim['Value']
                discovered['dimensions'].add(dim['Name'])

                if dim['Name'] == 'Operation':
                    discovered['operations'].add(dim['Value'])

            discovered['metrics'][metric_name].append(dim_combo)

        # Display summary
        print(f"📈 Unique Metrics: {', '.join(discovered['metrics'].keys())}")
        print(f"\n📐 Dimensions Used: {', '.join(discovered['dimensions'])}")

        if discovered['operations']:
            print(f"\n⚙️ Operations Found:")
            for op in sorted(discovered['operations']):
                print(f"   • {op}")

        # Show sample dimension combinations
        print(f"\n📊 Sample Metric Configurations:")
        for metric_name, combos in list(discovered['metrics'].items())[:3]:
            print(f"\n{metric_name}:")
            for combo in combos[:2]:
                dim_str = ', '.join([f"{k}={v.split('/')[-1] if '/' in v else v}"
                                    for k, v in combo.items() if k != 'Resource'])
                print(f"   • {dim_str if dim_str else 'Base metric'}")
            if len(combos) > 2:
                print(f"   ... and {len(combos) - 2} more combinations")

    except Exception as e:
        print(f"❌ Error: {str(e)}")

    return discovered

# Usage for all resources:
print("\n" + "="*60)
print("DISCOVERING ALL AGENTCORE METRICS")
print("="*60)

if resources.get('runtime_arn'):
    runtime_discovery = discover_all_metrics_for_resource(
        resources['runtime_arn'],
        "Runtime"
    )

if resources.get('memory_arn'):
    memory_discovery = discover_all_metrics_for_resource(
        resources['memory_arn'],
        "Memory"
    )

if resources.get('gateway_arn'):
    gateway_discovery = discover_all_metrics_for_resource(
        resources['gateway_arn'],
        "Gateway"
    )



DISCOVERING ALL AGENTCORE METRICS

🔍 Discovering all metrics for Runtime
🔗 Resource: arn:aws:bedrock-agentcore:us-east-1:533267284022:runtime/customer_support_agent-b0Ilb5ACG7
--------------------------------------------------
✅ Found 8 metric combinations

📈 Unique Metrics: Throttles, Duration, UserErrors, SystemErrors, Invocations, Latency, Sessions, Errors

📐 Dimensions Used: Resource, Name, Operation

⚙️ Operations Found:
   • InvokeAgentRuntime

📊 Sample Metric Configurations:

Throttles:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT

Duration:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT

UserErrors:
   • Operation=InvokeAgentRuntime, Name=customer_support_agent::DEFAULT

🔍 Discovering all metrics for Memory
🔗 Resource: arn:aws:bedrock-agentcore:us-east-1:533267284022:memory/CustomerSupportMemory-DB1nof41H6
--------------------------------------------------
✅ Found 18 metric combinations

📈 Unique Metrics: Latency, Invocations